In [ ]:
import orjson
import genson
import natsort
import polars
import rich
from data_index.iceberg_config import S3TablesCatalogConfig, IcebergTableConfig
import pyiceberg
import datetime

In [ ]:
# --- Sinks config ---
data_index_catalog_config = S3TablesCatalogConfig(
    region="ap-southeast-2",
    arn="arn:aws:s3tables:ap-southeast-2:704910415367:bucket/imos-data-inventory",
)

table = IcebergTableConfig(
    catalog_config=data_index_catalog_config,
    namespace="inventory",
    table_name=f"live",
).load()

In [ ]:
table.schema

In [ ]:
def get_lookback_timestamp(time_delta: datetime.timedelta) -> str:
    return datetime.datetime.combine(
        date=datetime.date.today() - time_delta, 
        time=datetime.time.min,
    ).isoformat()

get_lookback_timestamp(time_delta=datetime.timedelta(days=10))

In [ ]:
(
    table.scan(
        row_filter=f"last_modified_date >= '{get_lookback_timestamp(time_delta=datetime.timedelta(days=10))}'"
    )
    .to_polars()
)

In [ ]:
table.scan().to_polars()